# Weaveを活用したRAGの評価

## データセットの用意

In [ ]:
import os
from typing import TypedDict

import weave
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

weave.init(os.environ["WANDB_PROJECT"])

dataset = weave.Dataset(
    name="qa-basic",
    rows=[
        {
            "question": "介護給付費負担金の令和８年度の概算要求額は？",
            "expected": "2兆4,434億円です。",
            "source": "令和8年度概算要求の概要（老健局）の参考資料.pdf",
        },
        {
            "question": "健康日本２１（第２次）の目標は",
            "expected": "健康寿命の延伸と健康格差の縮小です。",
            "source": "健康日本21（第二次）参考資料スライド集.pptx",
        },
        {
            "question": "生涯現役促進地域連携事業の実施スキームで、協議会と厚労省はどんな関係にある？",
            "expected": "協議会が厚生労働省に企画提案を行い、厚生労働省が協議会を選抜して事業を委託します。連携・誘導は双方向です。",
            "source": "生涯現役促進地域連携事業の概要.pptx",
        },
    ],  # type: ignore
)
weave.publish(dataset)

## 評価の実行

### 評価対象のRAGの用意

In [ ]:
from pathlib import Path

import lancedb
from openai import OpenAI


class RagOutput(TypedDict):
    answer: str | None
    sources: list[str]


DB_DIR = Path("../data/lancedb")
client = OpenAI(max_retries=10)

# ベクトルデータベースへの接続
table = lancedb.connect(DB_DIR).open_table("simple_rag")


# 文字列をベクトルに変換(埋め込み)
def embed(texts: list[str]) -> list[list[float]]:
    res = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return [d.embedding for d in res.data]


# 質問文をベクトルにして、コサイン距離が近いチャンクの上位5件を取り出す
@weave.op
def search(question: str) -> list[dict[str, str]]:
    hits = table.search(embed([question])[0]).distance_type("cosine").limit(5).to_list()
    return [{"source": hit["source"], "text": hit["text"]} for hit in hits]


# LLMに、取り出したチャンクを情報として与えて質問に答えさせる
@weave.op
def rag(question: str) -> RagOutput:

    # 検索の実行
    hits = search(question)

    # LLMの呼び出し
    documents = "\n\n".join(f"## {hit['source']}\n{hit['text']}" for hit in hits)
    res = client.chat.completions.create(
        model="gpt-6-luna",
        messages=[
            {
                "role": "system",
                "content": "与えられた文書だけを根拠に、質問に答えてください。",
            },
            {"role": "user", "content": f"# 文書\n{documents}\n\n# 質問\n{question}"},
        ],
    )
    return {
        "answer": res.choices[0].message.content,
        "sources": [hit["source"] for hit in hits],
    }

### 評価関数（スコアラー）の実装

In [ ]:
from pydantic import BaseModel, Field


# 正解のファイルが、検索結果の上位 5 件に入っているか
@weave.op
def retrieval_hit(source: str, output: RagOutput) -> dict[str, bool]:
    return {"hit": source in output["sources"]}


# 次の answer_correct で使うLLMのStructured Outputs
class Judgement(BaseModel):
    correct: bool
    reason: str = Field(description="理由を一文で")


# 正解と同じ内容を答えているか
@weave.op
def answer_correct(question: str, expected: str, output: RagOutput) -> dict[str, bool | str]:

    # LLM-as-a-judge
    res = client.chat.completions.parse(
        model="gpt-6-luna",
        messages=[
            {
                "role": "system",
                "content": "回答が正解と同じ内容であれば correct を true に、誤りや欠落があれば false にしてください。表現の違いは問いません。",
            },
            {
                "role": "user",
                "content": f"# 質問\n{question}\n\n# 正解\n{expected}\n\n# 回答\n{output['answer']}",
            },
        ],
        response_format=Judgement,
    )
    return res.choices[0].message.parsed.model_dump()

### 評価の実行

In [ ]:
# 評価の実行
evaluation = weave.Evaluation(dataset=dataset, scorers=[retrieval_hit, answer_correct])
await evaluation.evaluate(rag)